<b> Computing the Pattern Correlation <b>

In [ ]:
#Weights

#OBS
# Get latitude (assumes same for all)
lat = Spatial_corr_JanR['latitude']
lon = Spatial_corr_JanR['longitude']
# Create 2D cosine latitude weights
weights_2d_obs = xr.DataArray(
    np.cos(np.deg2rad(lat)).broadcast_like(Spatial_corr_JanR),
    dims=["latitude", "longitude"],
    coords={"latitude": lat, "longitude": lon}
)

weights_2d_obs = weights_2d_obs.clip(min=0)

#ERA5
lat1 = Spatial_corr_JanR_Re['latitude']
lon1 = Spatial_corr_JanR_Re['longitude']
# Create 2D cosine latitude weights
weights_2d_era5 = xr.DataArray(
    np.cos(np.deg2rad(lat)).broadcast_like(Spatial_corr_JanR_Re),
    dims=["latitude", "longitude"],
    coords={"latitude": lat1, "longitude": lon1}
)

weights_2d_era5 = weights_2d_era5.clip(min=0)



#20CR
lat2 = Spatial_corr_JanR_Re20['latitude']
lon2 = Spatial_corr_JanR_Re20['longitude']
# Create 2D cosine latitude weights
weights_2d_20cr = xr.DataArray(
    np.cos(np.deg2rad(lat2)).broadcast_like(Spatial_corr_JanR_Re20),
    dims=["latitude", "longitude"],
    coords={"latitude": lat2, "longitude": lon2}
)

weights_2d_20cr = weights_2d_20cr.clip(min=0)

<b> Obs (Over Land) </b>

In [ ]:
# Define updated corr_maps for HD and HW (no suffix, with obs)
corr_maps_base = {
    'Jan': Spatial_corr_JanR, 'Feb': Spatial_corr_FebR, 'Mar': Spatial_corr_MarR,
    'Apr': Spatial_corr_AprR, 'May': Spatial_corr_MayR, 'Jun': Spatial_corr_JunR,
    'Jul': Spatial_corr_JulR, 'Aug': Spatial_corr_AugR, 'Sep': Spatial_corr_SepR,
    'Oct': Spatial_corr_OctR, 'Nov': Spatial_corr_NovR, 'Dec': Spatial_corr_DecR,
}

# Define updated corr_maps for CD and CW (with R1 suffix, with obs)
corr_maps_R1 = {
    'Jan': Spatial_corr_JanR1, 'Feb': Spatial_corr_FebR1, 'Mar': Spatial_corr_MarR1,
    'Apr': Spatial_corr_AprR1, 'May': Spatial_corr_MayR1, 'Jun': Spatial_corr_JunR1,
    'Jul': Spatial_corr_JulR1, 'Aug': Spatial_corr_AugR1, 'Sep': Spatial_corr_SepR1,
    'Oct': Spatial_corr_OctR1, 'Nov': Spatial_corr_NovR1, 'Dec': Spatial_corr_DecR1,
}

# Define ratio maps for each event type (no changes needed)
ratio_maps_all = {
    "HD": {
        'Jan': RatioJanHD, 'Feb': RatioFebHD, 'Mar': RatioMarHD, 'Apr': RatioAprHD,
        'May': RatioMayHD, 'Jun': RatioJunHD, 'Jul': RatioJulHD, 'Aug': RatioAugHD,
        'Sep': RatioSepHD, 'Oct': RatioOctHD, 'Nov': RatioNovHD, 'Dec': RatioDecHD,
    },
    "HW": {
        'Jan': RatioJanHW, 'Feb': RatioFebHW, 'Mar': RatioMarHW, 'Apr': RatioAprHW,
        'May': RatioMayHW, 'Jun': RatioJunHW, 'Jul': RatioJulHW, 'Aug': RatioAugHW,
        'Sep': RatioSepHW, 'Oct': RatioOctHW, 'Nov': RatioNovHW, 'Dec': RatioNovHW,
    },
    "CW": {
        'Jan': RatioJanCW, 'Feb': RatioFebCW, 'Mar': RatioMarCW, 'Apr': RatioAprCW,
        'May': RatioMayCW, 'Jun': RatioJunCW, 'Jul': RatioJulCW, 'Aug': RatioAugCW,
        'Sep': RatioSepCW, 'Oct': RatioOctCW, 'Nov': RatioNovCW, 'Dec': RatioDecCW,
    },
    "CD": {
        'Jan': RatioJanCD, 'Feb': RatioFebCD, 'Mar': RatioMarCD, 'Apr': RatioAprCD,
        'May': RatioMayCD, 'Jun': RatioJunCD, 'Jul': RatioJulCD, 'Aug': RatioAugCD,
        'Sep': RatioSepCD, 'Oct': RatioOctCD, 'Nov': RatioNovCD, 'Dec': RatioDecCD,
    }
}

months = list(corr_maps_base.keys())
results = {"Month": months}

for event_type, ratio_maps in ratio_maps_all.items():
    if event_type in ["HD", "HW"]:
        corr_maps = corr_maps_base
    else:  # CD, CW
        corr_maps = corr_maps_R1

    corrs = [
        round(xs.spearman_r(
            corr_maps[m],
            ratio_maps[m],
            dim=["latitude", "longitude"],
            weights=weights_2d_obs,
            skipna=True
        ).values.item(), 3)
        for m in months
    ]
    results[f"Obs({event_type})"] = corrs

df_all = pd.DataFrame(results)
print(df_all)

# Save to CSV if needed
#csv_path = "/g/data/gb02/nt4273/Confirmation_Plots/Obs_TP_LMF_ALL.csv"
#df_all.to_csv(csv_path, index=False)